In [18]:
import astropy.table as at
from astropy.table import QTable, Table, vstack, join
import numpy as np

In [29]:
def find_unimodal_hdbscan(cluster=None):
    # Load the unimodal results
    unimodal = Table.read('unimodalcheck_200M_phase.csv', format='csv')

    # Load the two cluster catalogs
    cat_6811 = QTable.read('rcat_ngc6811_v0.fits')
    cat_6866 = QTable.read('rcat_ngc6866_v0.fits')

    # Add a cluster label
    cat_6811['cluster'] = 'NGC6811'
    cat_6866['cluster'] = 'NGC6866'

    # Stack and keep unique stars
    catalog = vstack([cat_6811, cat_6866])
    catalog = at.unique(catalog, keys='GAIAEDR3_ID')

    # Rename id column to match catalog
    unimodal.rename_column('id', 'GAIAEDR3_ID')

    # Join on GAIAEDR3_ID
    matched = join(unimodal, catalog, keys='GAIAEDR3_ID', join_type='left')

    # Filter by cluster if specified
    if cluster == 'NGC6811':
        matched = matched[matched['cluster'] == 'NGC6811']
    elif cluster == 'NGC6866':
        matched = matched[matched['cluster'] == 'NGC6866']


    # Quick summary
    print(f"Cluster:                   {cluster if cluster else 'All'}")
    print(f"Total stars:               {len(matched)}")
    print(f"Unimodal stars:            {np.sum(matched['unimodal'] == 1)}")
    print(f"Non-unimodal stars:        {np.sum(matched['unimodal'] == 0)}")
    print(f"HDBscan_cluster=-2 stars:  {np.sum(matched['HDBscan_Cluster'] == -2)}")

    # Filter for unimodal AND HDBscan_Cluster == -2
    mask = (matched['unimodal'] == 1) & (['HDBscan_Cluster'] == -2)
    filtered = matched[mask]
    print(f"\nStars that are unimodal AND HDBscan_Cluster == -2: {len(filtered)}")

    unimodal_only = matched[matched['unimodal'] == 1]

    if len(unimodal_only) > 0:
        rand_id = np.random.choice(unimodal_only['GAIAEDR3_ID'])
        print("Random unimodal ID:", rand_id)
    else:
        print("No unimodal stars found")

In [30]:
find_unimodal_hdbscan(cluster='NGC6811')
find_unimodal_hdbscan(cluster='NGC6866')

Cluster:                   NGC6811
Total stars:               485
Unimodal stars:            16
Non-unimodal stars:        469
HDBscan_cluster=-2 stars:  9

Stars that are unimodal AND HDBscan_Cluster == -2: 0
Random unimodal ID: 2128173032662959232


Cluster:                   NGC6866
Total stars:               310
Unimodal stars:            5
Non-unimodal stars:        305
HDBscan_cluster=-2 stars:  3

Stars that are unimodal AND HDBscan_Cluster == -2: 0
Random unimodal ID: 2076067932542496000
